In [1]:
from pathlib import Path
from ultralytics import YOLO

ROOT = Path('/root/hit-uav')
DATA_YAML = ROOT / 'dataset.yaml'

model = YOLO('yolov8x.pt')  

# ============ 3. 微调训练 ============
results = model.train(
    data=str(DATA_YAML),
    epochs=100,             
    imgsz=640,              
    batch=16,               
    workers=8,
    device=0,               
    seed=0,                
    patience=30,            
    # ---- 优化器与学习率 ----
    optimizer='auto',          # 实际AdamW
    lr0=0.01,                 # 0.00125
    lrf=0.01,                 # 最终 = lr0 * lrf
    momentum=0.937,           #0.9
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=False,              # 余弦退火

    # ---- 预训练与冻结策略 ----
    pretrained=True,       
    freeze=None,           
                                           
    # ---- 数据增强（红外域重点）----
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,              # 航拍红外一般不做旋转
    translate=0.0,            # 轻微平移
    scale=0.0,                # 尺度抖动（强化小目标 bicycle/S 尺寸）
    fliplr=0.0,              # 左右翻转（目标检测常用）
    flipud=0.0,
    mosaic=0.0,               # Mosaic 
    mixup=0.0,                # 红外图 mixup 收益小，关闭
    close_mosaic=0,          # 最后 10 轮关闭 mosaic，稳定收敛

    # ---- 验证与输出 ----
    val=True,
    plots=True,
    name='yolov8x_finetune',   # 输出到 runs/detect/yolov8x_finetune_hituav
    project=str(ROOT / 'yolov8x_results'),
)

# ============ 4. 训练后最终验证 ============
metrics = model.val()
print(f"验证集 mAP50 = {metrics.box.map50:.4f}   mAP50-95 = {metrics.box.map:.4f}")

New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/hit-uav/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov

In [2]:
from pathlib import Path
import json
import numpy as np
from ultralytics import YOLO

DATA_ROOT = Path("/root/hit-uav")
IMG_DIR = DATA_ROOT / "images" / "test"
OUT_DIR = DATA_ROOT / "yolov8x_results" / "true_predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolov8x":  "/root/hit-uav/yolov8x_results/yolov8x_finetune/weights/best.pt",
}

files = sorted(IMG_DIR.glob("*.jpg"))
print(f"测试集图片数: {len(files)}")

for name, weight in MODELS.items():
    print(f"\n推理 {name} ...")
    model = YOLO(weight)
    pred_by_img = {}

    for f in files:
        r = model.predict(
            source=str(f),
            conf=0.001,       
            iou=0.7,           # NMS 阈值统一
            max_det=300,
            imgsz=640,
            verbose=False,
        )[0]

        dets = []
        if r.boxes is not None:
            for box, cls, conf_bit in zip(
                r.boxes.xyxy.tolist(),
                r.boxes.cls.tolist(),
                r.boxes.conf.tolist()
            ):
                dets.append([
                    int(cls),                      # HIT-UAV 类别索引
                    round(box[0], 2), round(box[1], 2),
                    round(box[2], 2), round(box[3], 2),
                    round(float(conf_bit), 6),
                ])
        pred_by_img[f.stem] = dets

    out_path = OUT_DIR / f"{name}_pred.json"
    json.dump(pred_by_img, open(out_path, "w"), ensure_ascii=False, indent=2)
    total = sum(len(v) for v in pred_by_img.values())
    print(f"{name}: {len(pred_by_img)} 张，总框数 {total}")
    print(f"保存: {out_path}")

测试集图片数: 571

推理 yolov8x ...
yolov8x: 571 张，总框数 6597
保存: /root/hit-uav/yolov8x_results/true_predictions/yolov8x_pred.json
